In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from enc_func import Dec, Enc_state, Enc_t, EncryptionConfig, Mod, Params, Seret_key
from FGH import FGHConfig, LQRConfig, PlantConfig, float_to_object_int, prepare_simulation_data


In [ ]:
ENC_CONFIG = EncryptionConfig(p=2**65, L=2**44, r=19.2, N=2**12, q_offset=31)
FGH_CONFIG = FGHConfig(
    plant=PlantConfig(J1=0.01, J2=0.01, J3=0.01, k1=1.37, k2=1.37, b1=0.007, b2=0.007, b3=0.007, Ts=0.1),
    lqr=LQRConfig(Q_diag=(1.0, 1.0, 1.0, 1.0, 1.0, 1.0), R_diag=(1.0,)),
)
r_quant = 100000
s_quant = 100000
iter_count = 10


In [ ]:
def attack_value(k, start, total):
    if k < start:
        return 0.0
    crit = (k - start) / total
    if crit < 0.1:
        return 1.0
    if 0.35 <= crit < 0.4:
        return -1.0
    return 0.0

env = Params(ENC_CONFIG)
sk = Seret_key(env)
prepared = prepare_simulation_data(env, s_quant=s_quant, config=FGH_CONFIG)
model, offline = prepared.model, prepared.offline
A, B, C, K = model.A, model.B.reshape(-1, 1), model.C, model.K.reshape(1, -1)
F_bar, G_bar, H_bar, Phi_pinv_bar = (offline[key] for key in ("F_bar", "G_bar", "H_bar", "Phi_pinv_bar"))
T1_all, T2_all, V2_all = (offline[key] for key in ("T1_all", "T2_all", "V2_all"))
S_xi_all, S_v_all, Psi_all, Sigma_all, Sigma_pinv_all = (offline[key] for key in ("S_xi_all", "S_v_all", "Psi_all", "Sigma_all", "Sigma_pinv_all"))

xp = [np.ones((6, 1), dtype=float)]
z_hat_bar = np.zeros((24, 1), dtype=int)
attack_start = iter_count // 2
attack_arr = np.zeros(iter_count)
u, y = [], []
residue = np.zeros((iter_count, 60))
execution_times = []
Z_hat_list, b_xi_list = [], []
for T1_j, T2_j, V2_j in zip(T1_all, T2_all, V2_all):
    Z_hat_j, b_xi_j = Enc_state(z_hat_bar, sk, env, T1_j, T2_j, V2_j)
    Z_hat_list.append(Z_hat_j)
    b_xi_list.append(b_xi_j)

x_hat_list = [Dec(Mod(Phi_pinv_bar @ Z_hat_list[0], env.q), sk, env) / (r_quant * s_quant * s_quant * env.L)]
channel_terms = [(H_bar[j : j + 1, :], S_xi_all[j], S_v_all[j], Psi_all[j], Sigma_all[j], Sigma_pinv_all[j]) for j in range(60)]

for k in tqdm(range(iter_count)):
    t_start = time.perf_counter()
    y_k = C @ xp[-1]
    u_k = (K @ xp[-1]).item()
    v = np.vstack([[[u_k]], y_k])
    attack = attack_value(k, attack_start, iter_count)
    attack_arr[k] = attack
    v[3, 0] += attack
    v_bar = float_to_object_int(v * r_quant)
    y.append(y_k)
    u.append(u_k)

    for j, (H_j, S_xi_j, S_v_j, Psi_j, Sigma_j, Sigma_pinv_j) in enumerate(channel_terms):
        Z_hat_j, b_xi_j = Z_hat_list[j], b_xi_list[j]
        r_bar_j = Mod(Mod(H_j @ Z_hat_j, env.q)[0, 0] * env.invL, env.q)
        residue[k, j] = float(r_bar_j) / (r_quant * s_quant * s_quant)
        V_j, b_v_j = Enc_t(v_bar, sk, b_xi_j, Sigma_pinv_j, Sigma_j, Psi_j, env)
        Z_hat_list[j] = Mod(F_bar @ Z_hat_j + G_bar @ V_j, env.q)
        b_xi_list[j] = Mod(S_xi_j @ b_xi_j + S_v_j @ b_v_j, env.q)

    x_hat_list.append(Dec(Mod(Phi_pinv_bar @ Z_hat_list[0], env.q), sk, env) / (r_quant * s_quant * s_quant * env.L))
    xp.append(A @ xp[-1] + B * u_k)
    execution_times.append(time.perf_counter() - t_start)

execution_times = np.array(execution_times)
print(f"Iteration time min  : {execution_times.min():.6f} s")
print(f"Iteration time max  : {execution_times.max():.6f} s")
print(f"Iteration time mean : {execution_times.mean():.6f} s")


In [ ]:
xp_arr = np.hstack(xp)
x_hat_arr = np.hstack(x_hat_list)
t = np.arange(iter_count)

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
axes[0].plot(t, np.max(np.abs(xp_arr[:, 1:] - x_hat_arr[:, 1:]), axis=0))
axes[0].axhline(0.05, color="red", linestyle="--", linewidth=1)
axes[0].grid(True)
axes[0].set_ylabel(r"$\|x_p(k)-\hat{x}(k)\|_\infty$")
axes[0].set_title("State estimation error")

axes[1].plot(t, attack_arr)
axes[1].grid(True)
axes[1].set_ylabel("attack")
axes[1].set_title("Injected Attack on sensor 3")

axes[2].plot(t, np.max(np.abs(residue), axis=1), label=r"$\|r\|_\infty$")
axes[2].axhline(0.1, color="red", linestyle="--", linewidth=1)
axes[2].grid(True)
axes[2].set_xlabel("time")
axes[2].set_ylabel(r"$\|r\|_\infty$")
axes[2].set_title("Residual infinity norm")
fig.tight_layout()
plt.show()
